## Delta Lake Setup (Storage & Notebook)
- *Notebook Setup:* Create a new notebook (e.g., `Delta tutorial`) and attach it to your Unity Catalog-enabled compute cluster.
- *Storage Setup (Important External Location Note):* If you create a brand-new container (e.g., `Delta Lake`) and try to create an external table in it, Databricks will throw a `"No parent external location was found"` error because your External Location is only configured for `mycontainer`.
- *Fix:* Inside your existing `mycontainer` in ADLS Gen2, add a new directory named `Delta Lake`.

## Creating an External Delta Table
- *Concept:* Create an external Delta table using the three-level namespace (`catalog.schema.table`), specifying `USING DELTA` and the ADLS Gen2 `LOCATION`.
- *SQL Example:*
```sql
CREATE TABLE man_cata.man_schema.delta_table (
    ID INT,
    name STRING,
    city STRING
)
USING DELTA
LOCATION 'abfss://mycontainer@storage_account_name.dfs.core.windows.net/Delta Lake/delta_table';

## Disabling Deletion Vectors (To Learn Traditional Delta Behavior)
- *Concept:* By default, Databricks auto-enables **Deletion Vectors** on new Delta tables. Turning them off first helps you understand traditional Delta Lake behavior (like file rewriting and tombstoning) before comparing it with modern Deletion Vectors.
- *SQL Example:*
```sql
ALTER TABLE man_cata.man_schema.delta_table
SET TBLPROPERTIES ('delta.enableDeletionVectors' = false);
```

## Inserting Data & Inspecting Metadata
- *Insert Data (SQL):* Populate the Delta table with sample records.
```sql
INSERT INTO man_cata.man_schema.delta_table
VALUES
    (1, 'AA', 'Delhi'),
    (2, 'BB', 'London'),
    (3, 'CC', 'Sydney');
```
- *Inspect Table Details (`DESCRIBE EXTENDED`):* Displays columns, data types, catalog, schema/database, table type (`EXTERNAL`), storage location, and provider (`delta`).
```sql
DESCRIBE EXTENDED man_cata.man_schema.delta_table;
```

## Delta Lake Architecture & `_delta_log`
- *Concept:* Delta Lake is **not** a separate file format; it is an **Open Table Format** (architecture) built on top of standard `.parquet` files.
- *How It Works:* When Databricks creates a `_delta_log` folder alongside your `.parquet` data files, it becomes a Delta table.
- *Inside `_delta_log`:*
  - **`.json` files (`000...0.json`, `1.json`, `2.json`):** Transaction logs stored in ascending order. Every operation (table creation, property change, `INSERT`, `UPDATE`) creates a new JSON version recording `add` or `remove` actions on Parquet files.
  - **`.crc` files (Cyclic Redundancy Check):** Used to verify data integrity and ensure files are not corrupted.

## Updates & Tombstoning of Files (Without Deletion Vectors)
- *SQL Example:*
```sql
UPDATE man_cata.man_schema.delta_table
SET city = 'Toronto'
WHERE ID = 1;
```
- *What Happens Behind the Scenes (Tombstoning):*
  - Instead of modifying the existing Parquet file in place, Delta Lake **rewrites** all records (with the updated value) into a brand-new Parquet file.
  - The old Parquet file is **not** physically deleted; it is **Tombstoned** (marked as "dead").
  - In the latest `_delta_log` JSON file, Delta Lake records a `remove` entry for the old file and an `add` entry for the new file so queries only read the latest version.

## Data Versioning (`DESCRIBE HISTORY`)
- *Concept:* Every transaction in the `_delta_log` creates a new version of the table (e.g., Version 0: Create, Version 1: Set Properties, Version 2: Insert, Version 3: Update).
- *SQL Example:*
```sql
DESCRIBE HISTORY man_cata.man_schema.delta_table;
```

## Time Travel (`RESTORE TABLE`)
- *Concept:* Because older Parquet files are retained (tombstoned) and tracked in the transaction log, you can roll a Delta table back to any previous version if a mistake occurs.
- *SQL Example:*
```sql
RESTORE TABLE man_cata.man_schema.delta_table TO VERSION AS OF 2;
```

## Deletion Vectors (Modern Delta Lake Updates)
- *Concept:* With Deletion Vectors enabled (default in modern Databricks), Delta Lake avoids rewriting entire Parquet files during `UPDATE` or `DELETE` operations.
- *Setup (New Table with Default Deletion Vectors Enabled):*
```sql
CREATE TABLE man_cata.man_schema.delta_table2 (
    ID INT,
    name STRING,
    city STRING
)
USING DELTA
LOCATION 'abfss://mycontainer@storage_account_name.dfs.core.windows.net/Delta Lake/delta_table2';

INSERT INTO man_cata.man_schema.delta_table2
VALUES
    (1, 'AA', 'Delhi'),
    (2, 'BB', 'London'),
    (3, 'CC', 'Sydney');

UPDATE man_cata.man_schema.delta_table2
SET city = 'Seattle'
WHERE ID = 1;
```
- *What Happens Behind the Scenes:*
  - Delta Lake creates a **Deletion Vector** file that flags the modified row (`ID = 1`) in the original Parquet file as removed/changed.
  - It writes **only the changed record** into a new, small Parquet file (instead of rewriting the whole file).
  - In the JSON transaction log, it removes the original entry, adds the new small file, and adds the original Parquet file back **linked with the Deletion Vector** so readers know which rows to skip.

## Optimizing Delta Tables (`OPTIMIZE`)
- *Concept:* Frequent updates and inserts create many small Parquet files (**small file problem**), which slows down read performance. `OPTIMIZE` compacts multiple small Parquet files into fewer, larger files.
- *Auto-Optimize with Deletion Vectors:* When Deletion Vectors are enabled, Delta Lake automatically runs `OPTIMIZE` in the background to compact the small files generated by updates (you can see this `OPTIMIZE` operation in `DESCRIBE HISTORY`).
- *Manual SQL Example:*
```sql
OPTIMIZE man_cata.man_schema.delta_table2;
```

## Deep Clone vs. Shallow Clone
- *Deep Clone:*
  - Copies **both metadata and actual data** (similar to `CTAS` - Create Table As Select).
  - Completely independent of the source table—subsequent updates to the source table do not affect the clone.
  - Works on both managed and external tables.
  - *SQL Example:*
```sql
CREATE TABLE man_cata.man_schema.deep_clone_table
DEEP CLONE man_cata.man_schema.delta_table;
```
- *Shallow Clone:*
  - Copies **only the metadata** (`_delta_log`) and points to the source table's existing data files (no data duplication).
  - *Important Limitation:* In Unity Catalog, Shallow Clone is **only supported when the source table is a Managed Table** (it will throw an error on external tables).
  - *SQL Example:*
```sql
CREATE TABLE man_cata.man_schema.shallow_table
SHALLOW CLONE man_cata.man_schema.man_table;
```